In [1]:
from pathlib import Path
import pandas as pd
import json

import sys
sys.path.append("../../../utils/")

from utils import *

In [2]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[3]

NOMBRE_DATASET_ENTRADA = "BCCC17__cleanning__v2"
NOMBRE_SPLIT = "BCCC17__split__v2"

RUTA_DATASET_LIMPIO = PROJECT_ROOT / "02_datasets" / "processed_analisis_estadistico" / NOMBRE_DATASET_ENTRADA
RUTA_SALIDA = PROJECT_ROOT / "02_datasets" / "processed_analisis_estadistico" / NOMBRE_SPLIT

NOMBRE_DATASET_LIMPIO = f"{NOMBRE_DATASET_ENTRADA}.csv"
NOMBRE_TRAIN = f"{NOMBRE_SPLIT}__train.csv"
NOMBRE_TEST = f"{NOMBRE_SPLIT}__test.csv"
NOMBRE_REPORTE = f"{NOMBRE_SPLIT}_report.json"

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"
TEST_SIZE = 0.20
RANDOM_STATE = 42

In [3]:
print("PROJECT_ROOT:")
print(PROJECT_ROOT)
print()

print("Dataset limpio de entrada:")
print(RUTA_DATASET_LIMPIO / NOMBRE_DATASET_LIMPIO)
print()

print("Ruta de salida:")
print(RUTA_SALIDA)

PROJECT_ROOT:
/LUSTRE/home/inginf/u32902122/TFG

Dataset limpio de entrada:
/LUSTRE/home/inginf/u32902122/TFG/02_datasets/processed_analisis_estadistico/BCCC17__cleanning__v2/BCCC17__cleanning__v2.csv

Ruta de salida:
/LUSTRE/home/inginf/u32902122/TFG/02_datasets/processed_analisis_estadistico/BCCC17__split__v2


In [4]:
input_path = RUTA_DATASET_LIMPIO / NOMBRE_DATASET_LIMPIO

if not input_path.exists():
    raise FileNotFoundError(f"No existe el dataset limpio en: {input_path}")

df = cargar_dataset(nombre_dataset=NOMBRE_DATASET_LIMPIO, ruta_base=RUTA_DATASET_LIMPIO)

shape_original = df.shape

print("Forma del dataset limpio:")
print(shape_original)

Forma del dataset limpio:
(2363699, 64)


In [5]:
df.head()

,DST_PORT,PROTOCOL,DURATION,PACKETS_COUNT,FWD_TOTAL_PAYLOAD_BYTES,PAYLOAD_BYTES_MAX,PAYLOAD_BYTES_MIN,PAYLOAD_BYTES_MEAN,PAYLOAD_BYTES_VARIANCE,FWD_PAYLOAD_BYTES_VARIANCE,...,BWD_SYN_FLAG_COUNTS,BWD_CWR_FLAG_COUNTS,BWD_RST_FLAG_COUNTS,PACKETS_IAT_MEAN,FWD_PACKETS_IAT_MEAN,FWD_PACKETS_IAT_STD,BWD_PACKETS_IAT_MEAN,SUBFLOW_FWD_PACKETS,SUBFLOW_FWD_BYTES,LABEL
0,8080,0,0.135358,8,194,194,0,40.25,5132.4375,7056.75,...,1,0,0,0.019337,0.044937,0.062157,0.044844,0.0,0.0,Botnet_ARES
1,8080,0,0.128585,8,194,194,0,40.25,5132.4375,7056.75,...,1,0,0,0.018369,0.042594,0.058929,0.042638,0.0,0.0,Botnet_ARES
2,8080,0,0.166355,10,194,194,0,32.50,4346.6500,6021.76,...,1,0,0,0.018484,0.041452,0.071034,0.041467,0.0,0.0,Botnet_ARES
3,8080,0,0.065549,8,1858,1858,0,248.25,371940.4375,647280.75,...,1,0,0,0.009364,0.021678,0.030040,0.021602,0.0,0.0,Botnet_ARES
4,8080,0,0.080872,8,194,194,0,40.25,5132.4375,7056.75,...,1,0,0,0.011553,0.026702,0.037218,0.026761,0.0,0.0,Botnet_ARES


In [6]:
if LABEL_COL not in df.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL}")

print("Columna objetivo encontrada correctamente.")
print()
print("Distribución global de clases:")
display(resumen_clases(df, LABEL_COL))

Columna objetivo encontrada correctamente.

Distribución global de clases:


,count,percentage
LABEL,,
Benign,1715048,72.5578
Botnet_ARES,5508,0.2330
DDoS_LOIT,95729,4.0500
DoS_GoldenEye,8364,0.3539
DoS_Hulk,346143,14.6441
DoS_Slowhttptest,6856,0.2901
DoS_Slowloris,5122,0.2167
FTP-Patator,9531,0.4032
Heartbleed,12,0.0005


In [7]:
train_df, test_df = dividir_train_test_stratified(
    df=df,
    label_col=LABEL_COL,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print("Forma train:", train_df.shape)
print("Forma test:", test_df.shape)

Forma train: (1890959, 64)
Forma test: (472740, 64)


In [8]:
print("Distribución de clases en TRAIN:")
display(resumen_clases(train_df, LABEL_COL))

Distribución de clases en TRAIN:


,count,percentage
LABEL,,
Benign,1372038,72.5578
Botnet_ARES,4406,0.2330
DDoS_LOIT,76583,4.0500
DoS_GoldenEye,6691,0.3538
DoS_Hulk,276914,14.6441
DoS_Slowhttptest,5485,0.2901
DoS_Slowloris,4098,0.2167
FTP-Patator,7625,0.4032
Heartbleed,10,0.0005


In [9]:
print("Distribución de clases en TEST:")
display(resumen_clases(test_df, LABEL_COL))

Distribución de clases en TEST:


,count,percentage
LABEL,,
Benign,343010,72.5579
Botnet_ARES,1102,0.2331
DDoS_LOIT,19146,4.0500
DoS_GoldenEye,1673,0.3539
DoS_Hulk,69229,14.6442
DoS_Slowhttptest,1371,0.2900
DoS_Slowloris,1024,0.2166
FTP-Patator,1906,0.4032
Heartbleed,2,0.0004


In [10]:
resumen_global = resumen_clases(df, LABEL_COL).rename(
    columns={"count": "global_count", "percentage": "global_percentage"}
)

resumen_train = resumen_clases(train_df, LABEL_COL).rename(
    columns={"count": "train_count", "percentage": "train_percentage"}
)

resumen_test = resumen_clases(test_df, LABEL_COL).rename(
    columns={"count": "test_count", "percentage": "test_percentage"}
)

comparacion = pd.concat([resumen_global, resumen_train, resumen_test], axis=1)

print("Comparación global / train / test:")
display(comparacion)

Comparación global / train / test:


,global_count,global_percentage,train_count,train_percentage,test_count,test_percentage
LABEL,,,,,,
Benign,1715048,72.5578,1372038,72.5578,343010,72.5579
Botnet_ARES,5508,0.2330,4406,0.2330,1102,0.2331
DDoS_LOIT,95729,4.0500,76583,4.0500,19146,4.0500
DoS_GoldenEye,8364,0.3539,6691,0.3538,1673,0.3539
DoS_Hulk,346143,14.6441,276914,14.6441,69229,14.6442
DoS_Slowhttptest,6856,0.2901,5485,0.2901,1371,0.2900
DoS_Slowloris,5122,0.2167,4098,0.2167,1024,0.2166
FTP-Patator,9531,0.4032,7625,0.4032,1906,0.4032
Heartbleed,12,0.0005,10,0.0005,2,0.0004


In [11]:
guardar_dataset_csv(
    df=train_df,
    nombre_archivo=NOMBRE_TRAIN,
    ruta=RUTA_SALIDA
)

guardar_dataset_csv(
    df=test_df,
    nombre_archivo=NOMBRE_TEST,
    ruta=RUTA_SALIDA
)

print("Train guardado en:")
print(RUTA_SALIDA / NOMBRE_TRAIN)
print()
print("Test guardado en:")
print(RUTA_SALIDA / NOMBRE_TEST)

Train guardado en:
/LUSTRE/home/inginf/u32902122/TFG/02_datasets/processed_analisis_estadistico/BCCC17__split__v2/BCCC17__split__v2__train.csv

Test guardado en:
/LUSTRE/home/inginf/u32902122/TFG/02_datasets/processed_analisis_estadistico/BCCC17__split__v2/BCCC17__split__v2__test.csv


In [12]:
reporte_split = {
    "dataset_entrada": NOMBRE_DATASET_LIMPIO,
    "label_column": LABEL_COL,
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "shape_global": {
        "rows": int(df.shape[0]),
        "cols": int(df.shape[1])
    },
    "shape_train": {
        "rows": int(train_df.shape[0]),
        "cols": int(train_df.shape[1])
    },
    "shape_test": {
        "rows": int(test_df.shape[0]),
        "cols": int(test_df.shape[1])
    },
    "class_distribution_global": {
        str(k): int(v) for k, v in df[LABEL_COL].value_counts(dropna=False).to_dict().items()
    },
    "class_distribution_train": {
        str(k): int(v) for k, v in train_df[LABEL_COL].value_counts(dropna=False).to_dict().items()
    },
    "class_distribution_test": {
        str(k): int(v) for k, v in test_df[LABEL_COL].value_counts(dropna=False).to_dict().items()
    }
}

report_path = RUTA_SALIDA / NOMBRE_REPORTE

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(reporte_split, f, indent=2, ensure_ascii=False)

print("Reporte guardado en:")
print(report_path)

Reporte guardado en:
/LUSTRE/home/inginf/u32902122/TFG/02_datasets/processed_analisis_estadistico/BCCC17__split__v2/BCCC17__split__v2_report.json


In [13]:
print("========== RESUMEN SPLIT ==========")
print(f"Dataset de entrada: {NOMBRE_DATASET_LIMPIO}")
print(f"Forma global: {df.shape}")
print(f"Forma train: {train_df.shape}")
print(f"Forma test: {test_df.shape}")
print(f"Test size: {TEST_SIZE}")
print(f"Random state: {RANDOM_STATE}")
print("===================================")

========== RESUMEN SPLIT ==========
Dataset de entrada: BCCC17__cleanning__v2.csv
Forma global: (2363699, 64)
Forma train: (1890959, 64)
Forma test: (472740, 64)
Test size: 0.2
Random state: 42
